## Pulling the bill data
JOKES can't do econ/recession stuff. Probably for the best. Going to keep econ indicators as controls (like COVID inflation/unemployment, etc.)

### TO DO
* Assign yearly unemployment, state GDP, general fund growth (maybe if I can find it; might be better to get income).
* Get gender and party data for 2026 legislators; make bipartisan variable
* Explore compiled bill data; how many bills in each subject, avg number of cosponsors, get idea of what to do.

### KEY VARIABLES
#### SIMPLE BILL MODEL
* Number of cosponsors
* Money appropriated sum OR if money was appropriated
* Number of substitutes
* Subject area (want to narrow from 21 to reduce multicolin)
* Year fixed effect (COVID?)

#### 2026 ONLY MODEL
* Gender
* Party (might not be worth doing; UT overwhelming republican)
* Bipartisan

#### ECON CONTROLLED MODEL
* Unemployment
* GDP (rate of change; pct)
* General Fund Growth

In [2]:
# importing packages
import requests
import pandas as pd
import json
import time
import numpy as np
import re

## Data Cleaning

In [3]:
# cleaning fred data for stitching to bills data

df_fred = pd.read_csv('annual.csv')

# renaming variables for easier use
new_name = [ 'unemp_rate', 'unemp_rate_chg', 'unemp_rate_pct', 'real_gdp', 'real_gdp_chg', 'real_gdp_pct']
old_name = ['LAUST490000000000003A', 'LAUST490000000000003A_CHG', 'LAUST490000000000003A_PCH', 
            'UTRGSP', 'UTRGSP_CHG', 'UTRGSP_PCH']

df_fred[new_name] = df_fred[old_name]
df_fred = df_fred.drop(columns = old_name)

# changing obsv_date to year
def get_year(obv_date):
    return int(obv_date[0:4].strip())
df_fred['observation_date'] = df_fred['observation_date'].apply(get_year)

df_fred

,observation_date,unemp_rate,unemp_rate_chg,unemp_rate_pct,real_gdp,real_gdp_chg,real_gdp_pct
0,2016,3.3,-0.2,-5.71429,162528.2,7096.7,4.56581
1,2017,3.1,-0.2,-6.06061,172075.0,9546.8,5.87393
2,2018,2.9,-0.2,-6.45161,182106.0,10031.0,5.82943
3,2019,2.5,-0.4,-13.79310,192760.6,10654.6,5.85077
4,2020,4.8,2.3,92.00000,195323.2,2562.6,1.32942
5,2021,2.8,-2.0,-41.66667,211289.5,15966.3,8.17430
6,2022,2.4,-0.4,-14.28571,217848.8,6559.3,3.10441
7,2023,2.7,0.3,12.50000,226322.1,8473.3,3.88953
8,2024,3.2,0.5,18.51852,234300.6,7978.5,3.52529


In [53]:
# cleaning monthly cpi data
df_cpi = pd.read_csv('monthly.csv')
df_cpi['year'] = df_cpi['observation_date'].apply(get_year)
df_cpi = df_cpi.rename(columns = {'CPIAUCNS': 'cpi_m'})


# getting annual avg
cpi_avg = {}
inflation = {2026: 1.0}
for year in np.arange(2016, 2026, 1):
    mask = df_cpi['year'] == year
    cpi_months = []
    for month in df_cpi.loc[mask, 'cpi_m']:
        if pd.isna(month): # 2025 is missing an entry October 2025
            continue
        cpi_months.append(month)
    avg = sum(cpi_months)/len(cpi_months)
    cpi_avg[int(year)] = round(avg, 3)
    
    # stitching cpi_avg to FRED (it is from FRED, but since it was monthly it was different)
    mask_fred = df_fred['observation_date'] == year
    df_fred.loc[mask_fred, 'cpi_year'] = cpi_avg[year]
    
# converting to inflation multiplier
for year in cpi_avg:
    current = 2025
    inflation[int(year)] = round(cpi_avg[current]/cpi_avg[year], 5)

    # stitching inflation to FRED again
    mask_fred = df_fred['observation_date'] == year
    df_fred.loc[mask_fred, 'inflation'] = inflation[year]

print(cpi_avg)
print(inflation)
df_fred

{2016: 240.007, 2017: 245.12, 2018: 251.107, 2019: 255.657, 2020: 258.811, 2021: 270.97, 2022: 292.655, 2023: 304.702, 2024: 313.689, 2025: 321.943}
{2026: 1.0, 2016: 1.34139, 2017: 1.31341, 2018: 1.28209, 2019: 1.25928, 2020: 1.24393, 2021: 1.18811, 2022: 1.10008, 2023: 1.05658, 2024: 1.02631, 2025: 1.0}


,observation_date,unemp_rate,unemp_rate_chg,unemp_rate_pct,real_gdp,real_gdp_chg,real_gdp_pct,cpi_year,inflation
0,2016,3.3,-0.2,-5.71429,162528.2,7096.7,4.56581,240.007,1.34139
1,2017,3.1,-0.2,-6.06061,172075.0,9546.8,5.87393,245.120,1.31341
2,2018,2.9,-0.2,-6.45161,182106.0,10031.0,5.82943,251.107,1.28209
3,2019,2.5,-0.4,-13.79310,192760.6,10654.6,5.85077,255.657,1.25928
4,2020,4.8,2.3,92.00000,195323.2,2562.6,1.32942,258.811,1.24393
5,2021,2.8,-2.0,-41.66667,211289.5,15966.3,8.17430,270.970,1.18811
6,2022,2.4,-0.4,-14.28571,217848.8,6559.3,3.10441,292.655,1.10008
7,2023,2.7,0.3,12.50000,226322.1,8473.3,3.88953,304.702,1.05658
8,2024,3.2,0.5,18.51852,234300.6,7978.5,3.52529,313.689,1.02631


In [63]:
ut_bills = pd.read_csv('utah_bills_final.csv')

# removing all resolutions (simple, joint, concurrent) to have dataset of only laws
laws = ['SJR', 'HJR', 'SCR', 'HCR', 'SR0', 'HR0']
df_bills = ut_bills.loc[~ut_bills['bill_id'].str[:3].isin(laws)].copy().reset_index(drop=True)
print(df_bills.shape, ut_bills.shape)


# cleaning NAs of subjects, committees, cosponsors, floor_sponsor, money_approrpiated
na_names = ['subjects', 'committees','co_sponsors', 'floor_sponsor', 'money_appropriated']
df_bills[na_names] = df_bills[na_names].replace({np.nan:'None'})

# fixing the NA in last_action

df_bills['last_action'] = df_bills['last_action'].replace({np.nan:'Senate/ received from House'})


# making lists of all action types

location1 = ['House/ filed', 'Senate/ filed', ''] # died in starting chamber (committee, house or senate)

location2 = ['House/ received from Senate', 
            'Senate/ received from House',
            'Senate/ 2nd Reading Calendar to Rules'] # died in non-originating chamber

location3 = ['Governor Line Item Veto',
            'Governor Vetoed',
            'Senate/ strike enacting clause',
            'Bill Failed Review - Returned to House'] # died in gov office (or post-Legislature)

location4 = ['LFA/ fiscal note publicly available',
            'LFA/ fiscal note sent to sponsor',
            'Became Law w/o Governor Signature',
            'Senate/ to Governor',
            'House/ to Governor',
            'House/ to Lieutenant Governor',
            'Senate/ to Lieutenant Governor',
            'Governor Declined to Sign',
            'Senate/ received enrolled bill from Printing',
            'House/ enrolled bill to Printing',
            'Senate/ enrolled bill to Printing',
            'Governor Signed'] # final location (aka passed)

# converting 'last_action' to ordinal range 'final_location' (god bless numpy select)
conditions = [df_bills['last_action'].isin(location1),
              df_bills['last_action'].isin(location2),
              df_bills['last_action'].isin(location3),
              df_bills['last_action'].isin(location4)]
choices = [1,2,3,4]

df_bills['location_stage'] = np.select(conditions, choices)


# one hot coding stage location
location_names = ['stage_1', 'stage_2', 'stage_3', 'stage_4']
dummies = pd.get_dummies(df_bills['location_stage'], prefix = 'stage').astype(int)
df_bills = pd.concat([df_bills, dummies], axis = 1)


# converting 'last_action' to 'passed' binary variable
df_bills['passed'] = df_bills['last_action'].map(lambda x: 1 if x in location4 else 0)




# grouping our 870 subjects (yes, 870) into 21 groups
subject_to_group = {
    # Criminal Justice
    'Crimes': 'cj_civ', 'Offenses': 'cj_civ', 'Criminal Procedure': 'cj_civ',
    'Criminal Code': 'cj_civ', 'Criminal Conduct': 'cj_civ', 'Criminal Identification': 'cj_civ',
    'Sentencing': 'cj_civ', 'Parole': 'cj_civ', 'Probation': 'cj_civ',
    'Expungement': 'cj_civ', 'Juvenile Justice': 'cj_civ', 'Juvenile Justice Services': 'cj_civ',
    'Peace Officers': 'cj_civ', 'Peace Officer': 'cj_civ', 'Courts': 'cj_civ',
    'Arrest and Detention': 'cj_civ', 'Bail\\Pretrial Detention': 'cj_civ',
    'Bail/Pretrial Detention': 'cj_civ', 'Plea Agreements': 'cj_civ',
    'Restitution': 'cj_civ', 'Recidivism': 'cj_civ', 'Homicide': 'cj_civ',
    'Theft': 'cj_civ', 'Assault and Battery': 'cj_civ', 'Vandalism': 'cj_civ',
    'Kidnapping': 'cj_civ', 'Prostitution': 'cj_civ', 'Gambling': 'cj_civ',
    'Gangs': 'cj_civ', 'Human Trafficking': 'cj_civ', 'Sexual Offenses': 'cj_civ',
    'Sex Offender Registry': 'cj_civ', 'Offender Registries': 'cj_civ',
    'Death Penalty': 'cj_civ', 'Insanity Defense': 'cj_civ',
    'Insanity Defense/Competency': 'cj_civ', 'Competency': 'cj_civ',
    'Forfeiture Procedure': 'cj_civ', 'Search and Seizure': 'cj_civ',
    'Protective Orders': 'cj_civ', 'Protective Orders/Stalking Injuctions': 'cj_civ',
    'Bureau of Criminal Identification': 'cj_civ', 'Bureau of Criminal Investigation': 'cj_civ',
    'Commission on Criminal and Juvenile Justice': 'cj_civ', 'Sentencing Commission': 'cj_civ',
    'Inmates': 'cj_civ', 'Correctional Facilities': 'cj_civ',
    'Department of Corrections': 'cj_civ', 'Youth Corrections': 'cj_civ',
    'Juvenile Records': 'cj_civ', 'Code of Criminal Procedure': 'cj_civ',
    'Board of Pardons and Parole': 'cj_civ', 'Indigent Defense': 'cj_civ',
    'Indigent Counsel': 'cj_civ', 'Drug Court': 'cj_civ',
    'Law Enforcement and Criminal Justice': 'cj_civ', 'Peace Officer Standards and Training': 'cj_civ',
    'Utah Highway Patrol': 'cj_civ', 'Fraud': 'cj_civ',
    'Punishments': 'cj_civ', 'Punishment': 'cj_civ',
    'Self Defense': 'cj_civ', 'Defenses': 'cj_civ',
    'Weapons': 'cj_civ', 'Graffiti': 'cj_civ',

    # Education K-12
    'K-12 Education': 'educ', 'School Finance': 'educ', 'School Safety': 'educ',
    'School Districts': 'educ', 'Special Education': 'educ',
    'Standards and Curriculum': 'educ', 'School Accountability': 'educ',
    'School Personnel': 'educ', 'School Fees': 'educ',
    'School Facilities': 'educ', 'Student Assessment': 'educ',
    'Student Discipline': 'educ', 'Student Privacy': 'educ',
    'Student Health and Safety': 'educ', 'Student health and safety': 'educ',
    'Local Boards of Education': 'educ', 'State Board of Education': 'educ',
    'Local Education Agencies (LEAs)': 'educ', 'Charter Schools': 'educ',
    'Home Schooling': 'educ', 'Prekindergarten': 'educ',
    'School Community Councils': 'educ', 'School community councils': 'educ',
    'State School Funding Distribution': 'educ', 'Public Education': 'educ',
    'Public Education Data and Reporting': 'educ', 'Education Grant Programs': 'educ',
    'Competency in Education': 'educ', 'Strategic Planning for Education': 'educ',
    'School and Institutional Trust Fund Office': 'educ',
    'School and Institutional Trust Land Administration': 'educ',
    'Online Learning and/or Technology': 'educ', 'Educational Telecommunications': 'educ',
    'Applied Technology Education': 'educ', 'Technical Colleges': 'educ',
    'Campus Safety': 'educ', 'On-campus Speech': 'educ',

    # Higher Education
    'Higher Education': 'educ', 'Colleges and Universities': 'educ',
    'Utah Board of Higher Education': 'educ', 'State Board of Regents': 'educ',
    'Utah System of Higher Education': 'educ', 'Scholarships and Financial Aid': 'educ',
    'Concurrent Enrollment': 'educ', 'Nontraditional Students': 'educ',
    'Nontraditional students': 'educ', 'Higher Education Students': 'educ',
    'Performance Funding': 'educ', 'Utech Board of Trustees': 'educ',
    'Utah Education Savings Plan': 'educ',

    # Health Care
    'Public Health': 'health', 'Mental Health': 'health', 'Medicaid': 'health',
    'Medicare': 'health', 'Health Insurance': 'health', 'Hospitals': 'health',
    'Telehealth': 'health', 'Substance Abuse': 'health', 'Substance Abuse and Mental Health': 'health',
    'Substance Use Disorder Treatment': 'health', 'Opioids': 'health',
    'Medical Cannabis': 'health', 'Pharmaceuticals': 'health', 'Pharmacies': 'health',
    'Pharmacists': 'health', 'Pharmacy Benefit Managers': 'health',
    'Health Care': 'health', 'Health Care Facilities': 'health',
    'Health Care Professionals': 'health', 'Health Care Providers': 'health',
    'Health Care Data and Informatics': 'health', 'Health Care Statistics': 'health',
    'Health Benefits and Claims': 'health', 'Benefits/Claims, Health': 'health',
    'Health': 'health', 'Health and Human Services': 'health',
    'Department of Health': 'health', 'Department of Health and Human Services': 'health',
    'Diseases': 'health', 'Disease Control and Prevention': 'health',
    'Diseases Control and Prevention': 'health', 'Immunizations': 'health',
    'Maternal and Child Health': 'health', 'Family Health and Preparedness': 'health',
    'Mental Health Facilities': 'health', 'Mental Health Professional': 'health',
    'Mental Health Professionals': 'health', 'Local Mental Health Authorities': 'health',
    'Local Substance Abuse Authorities': 'health', 'Suicide': 'health',
    'Nursing Homes': 'health', 'Skilled Nursing Facilities': 'health',
    'Ambulatory Surgical Centers': 'health', 'Community Health Clinics': 'health',
    'Utah State Hospital': 'health', 'Utah State Developmental Center': 'health',
    'Facility Licensing, Certification, and Resident Assessment': 'health',
    'Health Facility Administrators': 'health', 'Physicians and Surgeons': 'health',
    'Osteopathic Physicians and Surgeons': 'health', 'Physician Assistants': 'health',
    'Nurses': 'health', 'Dentists and Dental Hygienists': 'health',
    'Optometrists': 'health', 'Psychologists': 'health',
    'Physical Therapists': 'health', 'Occupational Therapists': 'health',
    'Speech-Language Pathologists and Audiologists': 'health', 'Respiratory Care Providers': 'health',
    'Midwives': 'health', 'Acupuncturists': 'health', 'Naturopathic Physicians': 'health',
    'Chiropractic Physicians': 'health', 'Podiatric Physicians': 'health',
    'Massage Therapists': 'health', 'Hearing Instrument Specialists': 'health',
    'Radiologic Technologists, Assistants, and Technicians': 'health',
    'Dietitians': 'health', 'Genetic Counselors': 'health',
    'Athletic Trainers': 'health', 'Medical Records': 'health',
    'Medical Malpractice': 'health', 'Medical Examiner': 'health',
    'Medical Fees and Service for Employment': 'health',
    'Medical Assistance Programs': 'health', 'Advance Health Care Directive': 'health',
    'Civil Commitment': 'health', 'Abortion': 'health',
    'Electronic Cigarettes': 'health', 'Tobacco': 'health',
    'Tobacco and Other Nicotine Products': 'health', 'Alcohol': 'health',
    'Alcoholic Beverages': 'health', 'Alcoholic Beverage Control': 'health',
    'Alcoholic Beverage Control Commission': 'health',
    'Department of Alcoholic Beverage Control': 'health',
    'Department of Alcoholic Beverage Services': 'health',
    'Drug and Alcohol Testing': 'health', 'Drugs and Alcohol, Workplace': 'health',
    'Controlled Substances': 'health', 'Controlled Substances Database': 'health',
    'Public Health Laboratory': 'health', 'Vital Statistics': 'health',
    'Emergency Medical Services': 'health', 'Emergency Medical Services and Preparedness': 'health',
    'Rural Health': 'health', 'Children\'s Health Insurance Program': 'health',
    'Rehabilitation Services': 'health', 'Adult Protective Services': 'health',
    'Long-term Care Ombudsman': 'health', 'Aging and Adult Services': 'health',
    'Aging Services': 'health', 'Insurance, Health': 'health',
    'Commercial Health Insurance\\Managed Care Contracts': 'health',
    'Public Employees\' Health Program': 'health',

    # Environment & Natural Resources
    'Air Quality': 'env_nr', 'Water Quality': 'env_nr',
    'Water': 'env_nr', 'Water Rights': 'env_nr',
    'Water Rights - Const. Art. XVII': 'env_nr',
    'Water Conservation': 'env_nr', 'Water and Irrigation': 'env_nr',
    'Water Utilities, Irrigation, and Sewer': 'env_nr',
    'Drinking Water': 'env_nr', 'Great Salt Lake': 'env_nr',
    'Utah Lake': 'env_nr', 'Mining': 'env_nr',
    'Mines and Mining': 'env_nr', 'Forestry and Fire': 'env_nr',
    'Forestry, Fire, and State Lands': 'env_nr',
    'Forests': 'env_nr', 'Wildlife': 'env_nr',
    'Environmental Quality': 'env_nr',
    'Environmental Response and Remediation': 'env_nr',
    'Department of Environmental Quality': 'env_nr',
    'Environment': 'env_nr', 'Natural Resources': 'env_nr',
    'Department of Natural Resources': 'env_nr',
    'Solid Waste': 'env_nr', 'Waste': 'env_nr',
    'Recycling': 'env_nr', 'Hazardous Materials': 'env_nr',
    'Hazardous Substances': 'env_nr',
    'Hazardous Materials Transportation': 'env_nr',
    'Underground Storage Tanks': 'env_nr',
    'Asbestos': 'env_nr', 'Emissions Control': 'env_nr',
    'Air': 'env_nr', 'Radiation': 'env_nr',
    'Pesticide & Pest Control': 'env_nr',
    'Pesticide and Pest Control': 'env_nr',
    'Reclamation': 'env_nr', 'Conservation District': 'env_nr',
    'Conservation Districts': 'env_nr',
    'State Lands': 'env_nr', 'Sovereign Lands': 'env_nr',
    'Trust Lands': 'env_nr', 'Public Lands': 'env_nr',
    'Geological Survey': 'env_nr', 'Oil and Gas': 'env_nr',
    'Fossil Fuels': 'env_nr', 'Natural Gas': 'env_nr',
    'Dams and Canals': 'env_nr', 'Weeds': 'env_nr',
    'Litter': 'env_nr', 'Noise Abatement': 'env_nr',
    'West Traverse Sentinel Landscape': 'env_nr',
    'Animal Health': 'env_nr', 'Animals': 'env_nr',

    # Transportation
    'Transportation': 'other', 'Motor Vehicles': 'other',
    'Roads/Highways': 'other', 'Highways': 'other',
    'Public Transit': 'other', 'Public Transit Districts': 'other',
    'Railroads': 'other', 'Bicycles': 'other',
    'Autonomous Vehicles': 'other', 'Aeronautics': 'other',
    'Airports': 'other', 'Boating': 'other',
    'Motor Carrier Regulation': 'other', 'Commercial Motor Vehicle Regulation': 'other',
    'Commercial Driver License (CDL)': 'other', 'Driver License': 'other',
    'Vehicle Registration': 'other', 'Motor Vehicle Insurance': 'other',
    'Motor Vehicle Equipment': 'other', 'Motor Vehicle Coverings': 'other',
    'Motor Vehicle Size Limits': 'other', 'Size Limits, Motor Vehicles': 'other',
    'Safety Inspection, Motor Vehicles': 'other', 'Vehicle Safety Equipment': 'other',
    'Off-highway Vehicles': 'other', 'Motorcycles': 'other',
    'Tow Trucks': 'other', 'Towing': 'other',
    'Traffic Violations': 'other', 'Driving Under the Influence (DUI)': 'other',
    'Seat Belt Laws': 'other', 'Seat Belts': 'other',
    'Helmets': 'other', 'Pedestrian Safety': 'other',
    'Speed Limits': 'other', 'Photo Radar': 'other',
    'Transportation Related Fees': 'other', 'Transportation Funding': 'other',
    'Transportation Fund': 'other', 'Transportation Commission': 'other',
    'Department of Transportation': 'other',
    'Transportation Investment Fund of 2005': 'other',
    'Transit Transportation Investment Fund': 'other',
    'Active Transportation Investment Fund': 'other',
    'Rural Transportation Investment Fund': 'other',
    'Active Transportation': 'other', 'Congestion Management': 'other',
    'Right of Way': 'other', 'License Plates': 'other',
    'Registration and Registration Fees': 'other',
    'Division of Motor Vehicles': 'other', 'Ports of Entry': 'other',
    'Unmanned Aircraft': 'other', 'Ridesharing and Shared Mobility': 'other',
    'Transportation Network Company': 'other',
    'Heber Valley Historic Railroad Authority': 'other',
    'Utah State Railroad Museum Authority': 'other',
    'Equipment, Motor Vehicles': 'other', 'Trails': 'other',

    # Elections & Government Operations
    'Elections': 'gov_ops', 'Election Law': 'gov_ops',
    'Election Administration': 'gov_ops', 'Voting and Voter Registration': 'gov_ops',
    'Voter Registration': 'gov_ops', 'Campaign Finance': 'gov_ops',
    'Redistricting': 'gov_ops', 'Political Parties': 'gov_ops',
    'Initiatives': 'gov_ops', 'Referenda': 'gov_ops',
    'Initiatives / Referendums': 'gov_ops', 'Initiatives \\ Referendums': 'gov_ops',
    'Legislative Operations': 'gov_ops', 'Legislative Organization': 'gov_ops',
    'Legislature': 'gov_ops', 'Legislative Staff Offices': 'gov_ops',
    'Legislative Affairs': 'gov_ops', 'Legislative Committees and Task Forces': 'gov_ops',
    'Committees, Legislative': 'gov_ops', 'Legislative Employees and Compensation': 'gov_ops',
    'Employees and Compensation, Legislative': 'gov_ops',
    'Legislative Publications': 'gov_ops', 'Legislative Oversight of Administrative Rules': 'gov_ops',
    'Legislative Water Development Commission': 'gov_ops',
    'Administrative Rules': 'gov_ops', 'Administrative Rulemaking': 'gov_ops',
    'Administrative Rulemaking and Procedures': 'gov_ops',
    'Administrative Law': 'gov_ops', 'Administrative Law Judges': 'gov_ops',
    'Administrative Procedures': 'gov_ops', 'Administrative Services': 'gov_ops',
    'Rulemaking Procedures': 'gov_ops', 'New Rulemaking Authority': 'gov_ops',
    'Office of Administrative Rules': 'gov_ops',
    'Judicial Review of Administrative Rules': 'gov_ops',
    'Judicial Review of State Agency Adjudicative Proceedings': 'gov_ops',
    'State Agency Adjudicative Proceedings': 'gov_ops',
    'State Agency Review of Adjudicative Proceedings': 'gov_ops',
    'Governor': 'gov_ops', 'Lieutenant Governor': 'gov_ops',
    'Attorney General': 'gov_ops', 'State Auditor': 'gov_ops',
    'State Treasurer': 'gov_ops', 'Constitutional Officers': 'gov_ops',
    'Elected Official': 'gov_ops', 'Public Officers': 'gov_ops',
    'County Officers': 'gov_ops', 'Municipal Officers': 'gov_ops',
    'Ethics': 'gov_ops', 'Conflicts of Interest': 'gov_ops',
    'Lobbying': 'gov_ops', 'Government Records': 'gov_ops',
    'Public Meetings': 'gov_ops', 'Open and Public Meetings': 'gov_ops',
    'Governance': 'gov_ops', 'Federalism': 'gov_ops',
    'Federal Government': 'gov_ops', 'Interlocal Cooperation': 'gov_ops',
    'Interlocal Agreements': 'gov_ops', 'Interlocal Entities': 'gov_ops',
    'Interlocal Cooperation Entities': 'gov_ops',
    'Task Force / Committees': 'gov_ops', 'Boards and Committees': 'gov_ops',
    'State Boards, Commissions, and Councils': 'gov_ops',
    'Revisor Legislation': 'gov_ops', 'Recodification': 'gov_ops',
    'Statutory Construction': 'gov_ops', 'Uniform Laws': 'gov_ops',
    'Resolutions': 'gov_ops', 'Resolutions, Rules': 'gov_ops',
    'Sunset Legislation': 'gov_ops', 'Sunsets and Repealers': 'gov_ops',
    'Boxcar Legislation': 'gov_ops', 'State Affairs in General': 'gov_ops',
    'Publications': 'gov_ops', 'Publications and Broadcasting': 'gov_ops',
    'Official Notices': 'gov_ops', 'Notice Requirements': 'gov_ops',
    'Archives': 'gov_ops', 'Archives and Records': 'gov_ops',
    'History': 'gov_ops', 'Capitol Hill': 'gov_ops',
    'State Symbols and Designations': 'gov_ops',
    'Flag (American)': 'gov_ops', 'Constitution': 'gov_ops',
    'Utah Constitutional Amendments': 'gov_ops',
    'Independent Entities': 'gov_ops',
    'Retirement and Independent State Entities': 'gov_ops','Governmental Immunity': 'gov_ops',
    'Tribal Nations Within Utah': 'gov_ops','Division of Indian Affairs': 'gov_ops',
    'Repatriation of Native American Remains': 'gov_ops',

    # Taxation & Finance
    'Income Tax': 'gov_ops', 'Individual Income Tax': 'gov_ops',
    'Sales Tax Exemptions': 'gov_ops', 'Sales and Use Tax': 'gov_ops',
    'Property Tax': 'gov_ops', 'Property Tax Collection': 'gov_ops',
    'Property Tax Relief': 'gov_ops', 'Farmland Assessment': 'gov_ops',
    'Assessment and Equalization': 'gov_ops', 'Truth in Taxation': 'gov_ops',
    'Tax Credits': 'gov_ops', 'Tax Increment Financing': 'gov_ops',
    'Excise Taxes': 'gov_ops', 'Corporate Tax': 'gov_ops',
    'Severance Tax': 'gov_ops', 'Privilege Tax': 'gov_ops',
    'Gross Receipts': 'gov_ops', 'Motor Fuel and Special Fuel Taxes': 'gov_ops',
    'Cigarette and Tobacco Taxes': 'gov_ops', 'Beer Tax': 'gov_ops',
    'Local Option Sales Taxes': 'gov_ops', 'Local Taxation and Fees': 'gov_ops',
    'Tribal Tax Issues': 'gov_ops', 'Revenue': 'gov_ops',
    'Revenue and Taxation': 'gov_ops', 'State Tax Commission': 'gov_ops',
    'Appropriations': 'gov_ops', 'Budgeting': 'gov_ops',
    'Public Budgeting': 'gov_ops', 'Fiscal Procedures': 'gov_ops',
    'Finance': 'gov_ops', 'Division of Finance': 'gov_ops',
    'Bonds': 'gov_ops', 'Public Bonds': 'gov_ops',
    'Public Funds and Accounts': 'gov_ops', 'Fees': 'gov_ops',
    'Uniform Fees': 'gov_ops', 'Fines': 'gov_ops',
    'Local Expenditures': 'gov_ops', 'Unclaimed Property': 'gov_ops',
    'Mineral Lease Funds': 'gov_ops', 'Permanent Community Impact Fund': 'gov_ops',
    'Impact Fees': 'gov_ops', 'Assessment Area': 'gov_ops',
    'Energy Assessment Area': 'gov_ops', 'County and Municipal Finance': 'gov_ops',
    'State Infrastructure Bank': 'gov_ops', 'Utah Capital Investment Corporation': 'gov_ops',
    'Risk Management Fund': 'gov_ops', 'Division of Risk Management': 'gov_ops',
    'Procurement': 'gov_ops', 'Government Purchasing': 'gov_ops',
    'Division of Purchasing and General Services': 'gov_ops',

    # Labor & Employment
    'Labor': 'econ', 'Labor & Employment': 'econ',
    'Labor and Employment': 'econ', 'Labor Commission': 'econ',
    'Labor Disputes': 'econ', 'Labor Organizations': 'econ',
    'Working Conditions': 'econ', 'Workplace Safety and Health': 'econ',
    'Occupational Safety and Health (OSHA)': 'econ',
    'Occupational Disease': 'econ', 'Industrial Accident': 'econ',
    'Workers\' Compensation': 'econ', 'Unemployment Insurance': 'econ',
    'Wages': 'econ', 'Right to Work': 'econ',
    'Collective Bargaining': 'econ', 'Unfair Labor Practices': 'econ',
    'Employment Agencies': 'econ', 'Employment of Minors': 'econ',
    'Apprenticeship Training': 'econ', 'Job Training': 'econ',
    'Workforce Services': 'econ', 'Department of Workforce Services': 'econ',
    'Department of Human Resource Management': 'econ',
    'Division of Human Resource Management': 'econ',
    'State Officers and Employees': 'econ',
    'Local Government Employees': 'econ',
    'Sexual Harassment': 'econ', 'Antidiscrimination': 'econ',

    # Family & Social Services
    'Child Welfare': 'health', 'Foster Care': 'health',
    'Adoption': 'health', 'Child Custody/Parent Time': 'health',
    'Child Custody \\ Parent-Time': 'health',
    'Child Support': 'health', 'Child Care': 'health',
    'Child Care Licensing': 'health', 'Child \\ Day Care': 'health',
    'Child \\ Day Care Licensing': 'health',
    'Child Development': 'health', 'Children': 'health',
    'Minors': 'health', 'Juveniles': 'health',
    'Domestic Violence': 'health', 'Family': 'health',
    'Marriage / Divorce': 'health', 'Marriage/Divorce': 'health',
    'Parentage': 'health', 'Grandparents': 'health',
    'Parental Defense': 'health', 'Guardian Ad Litem': 'health',
    'Guardianship/Conservatorship': 'health', 'Guardians': 'health',
    'Conservators': 'health', 'Emancipation': 'health',
    'Homelessness': 'health', 'Homeless Persons': 'health',
    'Intergenerational Poverty': 'health', 'Public Assistance': 'health',
    'Human Services': 'health', 'Human Services Programs': 'health',
    'Human Services Licensure': 'health', 'Human Services Facilities': 'health',
    'Department of Human Services': 'health',
    'Adult Services': 'health', 'Elderly': 'health',
    'Division of Child and Family Services': 'health',
    'Office of Recovery Services': 'health',
    'Abuse, Neglect, Exploitation of Vulnerable Adults': 'health',
    'Abuse, Neglect, or Exploitation of Vulnerable Adults': 'health',
    'Abuse, Neglect, or Dependency': 'health',
    'Victims': 'health', 'Victims\' Rights': 'health',
    'Disabilities': 'health', 'Services for People with Disabilities': 'health',
    'Blind Persons': 'health', 'Blind and Visually-impaired Services': 'health',
    'Utah Schools for the Deaf and the Blind': 'health',
    'Deaf and Hard of Hearing': 'health',
    'Deaf and Hard of Hearing Services': 'health',
    'Refugees': 'health', 'Immigrants \\ Immigration': 'health',
    'Immigration': 'health', 'Indian Affairs': 'health',
    'Multicultural Affairs': 'health',
    'Food Security': 'health', 'Fatality Reports': 'health',
    'Death': 'health', 'Licensure, Human Services Programs': 'health',

    # Housing & Land Use
    'Affordable Housing': 'econ', 'Housing': 'econ',
    'Housing and Community Development': 'econ',
    'Utah Housing Corporation': 'econ', 'Planning and Zoning': 'econ',
    'Real Estate': 'econ', 'Real Property': 'econ',
    'Landlord - Tenant': 'econ', 'Landlord -- Tenant': 'econ',
    'Land Use': 'econ', 'Condominiums': 'econ',
    'Community/Condominium Associations (HOAs)': 'econ',
    'Eminent Domain': 'econ',
    'Eminent Domain (Government Land Take Over)': 'econ',
    'Annexation': 'econ', 'Municipal Boundaries': 'econ',
    'Local Government Boundaries': 'econ',
    'Incorporation': 'econ', 'Unincorporated Areas': 'econ',
    'Fair Housing': 'econ', 'Easements': 'econ',
    'Property Conveyance': 'econ', 'Mortgage': 'econ',
    'Mortgage/Deed of Trust': 'econ', 'Appraisals': 'econ',
    'Title and Escrow': 'econ', 'Liens': 'econ',
    'Construction': 'econ', 'Construction Industries': 'econ',
    'Construction and Fire Codes': 'econ',
    'State Buildings': 'econ', 'State Facilities': 'econ',
    'Facilities, Construction, and Management': 'econ',
    'Local Building Authority': 'econ',
    'Community Development': 'econ',
    'Community Development and Renewal Agencies': 'econ',
    'Community Reinvestment Agencies': 'econ',

    # Business & Commerce
    'Business': 'econ', 'Commerce and Trade': 'econ',
    'commerce': 'econ', 'Corporations': 'econ',
    'Limited Liability Company': 'econ', 'Limited Liability Companies': 'econ',
    'Partnerships': 'econ', 'Business Entities': 'econ',
    'Business Licensing': 'econ', 'Business Development': 'econ',
    'Contracts': 'econ', 'Contracts and Obligations': 'econ',
    'Consumer Protection': 'econ', 'Consumer Credit': 'econ',
    'Credit': 'econ', 'Antitrust': 'econ',
    'Antitrust Law': 'econ', 'Insurance': 'econ',
    'Insurance Department': 'econ', 'Department of Insurance': 'econ',
    'Property and Casualty Insurance': 'econ',
    'Life Insurance and Annuities': 'econ',
    'Motor Vehicle Insurance': 'econ',
    'Occupational and Professional Licensing': 'econ',
    'Division of Professional Licensing': 'econ',
    'Office of Licensing': 'econ',
    'Office of Licensing and Background Checks': 'econ',
    'Occupations and Professions': 'econ',
    'Securities': 'econ', 'Bankruptcy': 'econ',
    'Franchise': 'econ', 'Franchises': 'econ',
    'Retail Trade': 'econ', 'Manufacturing': 'econ',
    'Advertising': 'econ', 'Trademarks': 'econ',
    'International Trade': 'econ', 'Tourism': 'econ',
    'Department of Commerce': 'econ',
    'Department of Financial Institutions': 'econ',
    'Financial Institutions': 'econ', 'Banks': 'econ',
    'Credit Unions': 'econ', 'Charities': 'econ',
    'Governmental Nonprofit Corporations': 'econ',
    'Collection Agencies': 'econ', 'Pawnshops and Secondhand Merchandise': 'econ',
    'Rental and Leasing Services': 'econ',
    'Repair and Maintenance Services': 'econ',
    'Food Services and Drinking Places': 'econ',
    'Hotels and Hotel Keepers': 'econ', 'Warehousing and Storage': 'econ',
    'Personal Services': 'econ', 'Process Servers': 'econ',
    'Security Licensing': 'econ', 'Uniform Commercial Code': 'econ',
    'Electronic Transactions': 'econ', 'Negotiable Certificates': 'econ',
    'Product Liability': 'econ', 'Liability': 'econ',

    # Public Safety & Emergency Management
    'Public Safety': 'other', 'Fire Protection': 'other',
    'Fire Control': 'other', 'Emergency Management': 'other',
    'Emergency Powers': 'other', 'Homeland Security': 'other',
    'National Guard': 'other', 'Utah National Guard': 'other',
    'Military Services': 'other',
    'Department of Public Safety': 'other',
    'Veterans and Military Benefits': 'other',
    'Veterans and Military Affairs': 'other',
    'Veterans Affairs': 'other', 'Veterans\' Affairs': 'other',
    'Veterans Service Organizations': 'other',
    'Utah Veterans Advisory Council': 'other',
    'Department of Veterans and Military Affairs': 'other',
    'Military Installation Development Authority': 'other',
    'Miners': 'other', 'Background Checks': 'other',
    'Crisis Line': 'other',
    'Public Safety Retirement': 'other',
    'Firefighters\' Retirement': 'other',

    # Technology & Data
    'Technology': 'econ', 'Technology Governance': 'econ',
    'Department of Technology Services': 'econ',
    'Division of Technology Services': 'econ',
    'Agency Technology Services': 'econ',
    'Agency Technology Services and Public Websites': 'econ',
    'Information Technology Rate Committee': 'econ',
    'Artificial Intelligence': 'econ', 'Data and Cyber Security': 'econ',
    'Electronic Privacy': 'econ', 'Electronic Information': 'econ',
    'Electronic Databases': 'econ', 'Internet': 'econ',
    'Internet Protocols': 'econ', 'Internet Service Provider': 'econ',
    'Broadband': 'econ', 'Telecommunications': 'econ',
    'Telephone': 'econ', 'Social Media': 'econ',
    'Blockchain': 'econ', 'Management Information System': 'econ',
    'Automated Geographic Reference Center': 'econ',
    'Utah Geospatial Resource Center': 'econ',
    'Recordings': 'econ', 'News Agencies': 'econ',

    # Civil Rights & Liberties
    'Civil Rights': 'cj_civ', 'Antidiscrimination': 'cj_civ',
    'Fair Housing': 'cj_civ', 'Sexual Harassment': 'cj_civ',
    'Pornography': 'cj_civ',
    'Freedom of Speech - Const. Art. I, Sec 15': 'cj_civ',
    'On-campus Speech': 'cj_civ', 'Civil Litigation': 'cj_civ',
    'Civil Procedure': 'cj_civ', 'Statute of Limitations': 'cj_civ',
    'Subpoenas': 'cj_civ', 'Juries': 'cj_civ',
    'Appeals': 'cj_civ', 'Court Rules': 'cj_civ',
    'Court Procedure': 'cj_civ', 'Judicial Administration': 'cj_civ',
    'Judicial Operations': 'cj_civ', 'Judiciary': 'cj_civ',
    'Judges': 'cj_civ', 'Attorneys': 'cj_civ',
    'Alternative Dispute Resolution': 'cj_civ',
    'Arbitration\\Mediation': 'cj_civ',
    'Dispute Resolution': 'cj_civ',
    'Notarization and Authentication': 'cj_civ',

    # Energy
    'Energy': 'other', 'Renewable Energy': 'other', 'Renewable and Clean Energy': 'other',
    'Clean Energy': 'other', 'Clean Fuels': 'other', 'Solar Power': 'other',
    'Wind Power': 'other', 'Nuclear Energy': 'other', 'Thermal Power': 'other',
    'Electricity': 'other', 'Electric Cooperatives': 'other',
    'Investor-owned Utilities': 'other', 'Public Utilities': 'other',
    'Public Utilities & Technology': 'other', 'Public Utilities and Technology': 'other',
    'Public Service Commission': 'other', 'Division of Public Utilities': 'other',
    'Utilities Siting': 'other', 'Utilities Siting and Permitting': 'other',
    'Utility Programs and Energy Procurement': 'other',
    'Utah Energy Infrastructure Authority': 'other',
    'Office of Energy Development': 'other',
    'Governor\'s Office of Energy Development': 'other', 'Energy Storage': 'other',

    # Local Government
    'Counties': 'gov_ops', 'Municipalities': 'gov_ops',
    'Municipal Government': 'gov_ops', 'County Government': 'gov_ops',
    'Local Governments': 'gov_ops', 'Local Government Ordinances': 'gov_ops',
    'Local Government Infrastructure and Utilities': 'gov_ops',
    'Local Government Controlled Districts': 'gov_ops',
    'Local Governments Controlled Districts': 'gov_ops',
    'Local Government Classification': 'gov_ops',
    'Local Government Employees': 'gov_ops',
    'Special Service District': 'gov_ops', 'Special Service Districts': 'gov_ops',
    'Limited Purpose Local Government Entities': 'gov_ops',
    'Local Districts': 'gov_ops', 'Metro Townships': 'gov_ops',
    'Townships': 'gov_ops', 'Form of Local Government': 'gov_ops',
    'Community Councils': 'gov_ops', 'Political Subdivisions (Local Issues)': 'gov_ops',
    'Public Infrastructure District': 'gov_ops',
    'Rural Development': 'gov_ops',

    # Economic Development
    'Economic Development': 'econ',
    'Office of Economic Development': 'econ',
    'Governor\'s Office of Economic Development': 'econ',
    'Governor\'s Office of Economic Opportunity': 'econ',
    'Business Development': 'econ', 'Science and Biotechnology': 'econ',
    'Olympics': 'econ', 'Compacts': 'econ',
    'International Trade': 'econ',

    # Retirement & Benefits
    'Retirement': 'other', 'Utah Retirement Systems (URS)': 'other',
    'Public Employees Retirement': 'other',
    'Tier 1 Retirement': 'other', 'Tier 2 Retirement': 'other',
    'Constitutional Officer and Legislator Retirement': 'other',
    'Judges\' Retirement': 'other',
    'Public Safety Retirement': 'other',
    'Private Employer Retirement': 'other',
    'Public Employees Long-term Disabilities Benefits': 'other',
    'Public Employees Insurance and Benefits': 'other',
    'Public Retirement and Insurance': 'other',
    'Veterans and Military Benefits': 'other',

    # Agriculture & Food
    'Agriculture': 'env_nr', 'Agriculture and Food': 'env_nr',
    'Agriculture & Food': 'env_nr', 'Department of Agriculture and Food': 'env_nr',
    'Agriculture Inspections': 'env_nr', 'Livestock': 'env_nr',
    'Animal Food Products': 'env_nr', 'Plant Food Products': 'env_nr',
    'Plant Industry': 'env_nr', 'Food': 'env_nr',
    'Food Quality': 'env_nr', 'Dairy': 'env_nr',
    'Utah Dairy Commission': 'env_nr', 'Aquaculture and Game Ranching': 'env_nr',
    'Game Ranching': 'env_nr', 'Fishing': 'env_nr',
    'Hunting': 'env_nr', 'Veterinary Care': 'env_nr',
    'Dogs': 'env_nr',

    # Arts, Culture & Recreation
    'Arts': 'other', 'Heritage': 'other',
    'Heritage and Arts': 'other',
    'Department of Heritage and Arts': 'other',
    'Department of Cultural and Community Engagement': 'other',
    'Cultural and Community Engagement': 'other',
    'Cultural Engagement': 'other', 'Museums': 'other',
    'Libraries': 'other', 'Recreation': 'other',
    'Parks and Recreation': 'other',
    'State Parks and Recreation': 'other',
    'Division of State Parks': 'other',
    'Division of Recreation': 'other',
    'Outdoor Recreation': 'other', 'Parks': 'other',
    'Olympics': 'other', 'Athletics': 'other',
    'Motion Pictures': 'other',

    # No Subjects
    'None': 'other',

}

# getting the most frequent subject group from each bill
from collections import Counter

def get_primary_group(subject_object):
    if pd.isna(subject_object):
        return 'other'
    subjects = [sub.strip() for sub in subject_object.split('; ')]
    groups = [subject_to_group[subj] for subj in subjects if subj in subject_to_group]
    if not groups:
        return 'other'
    return Counter(groups).most_common(1)[0][0]

df_bills['subject_group'] = df_bills['subjects'].apply(get_primary_group)

# making a column for each subject type
col_names = list(set(subject_to_group.values()))

for col in col_names:
    df_bills[col] = df_bills['subject_group'].map(lambda x: 1 if x == col else 0)



# making one-hot dummy variables (committee, cosponsor, floor sponsor, money appropriated
dummy_mask = ['committees','co_sponsors', 'floor_sponsor', 'money_appropriated']
dummy_name = ['committee1', 'cospon1', 'floorspon1', 'money1']
df_bills[dummy_name] = df_bills[dummy_mask].map(lambda x: 0 if x == 'None' else 1)




# get # of cosponsors variable
def cospon_sum(subject_object):
    if pd.isna(subject_object) or subject_object == 'None':
        return 0
    cospon = [c for c in subject_object.split('; ')]
    return len(cospon)

df_bills['cospon_num'] = df_bills['co_sponsors'].apply(cospon_sum)




# update 'num_substitutes' to align with active version (pre-2025); new column 'num_sub_new'
df_bills['num_sub1'] = np.where(
    df_bills['session_year'] < 2025,
    df_bills['active_version'],
    df_bills['num_substitutes'] )
df_bills.tail()

def sub_num(subject_object):
    if pd.isna(subject_object):
        return 0
    if isinstance(subject_object, int):
        return subject_object
    if subject_object[-3] == 'S':
        return int(subject_object[-1])
    else:
        return 0

df_bills['num_sub_new'] = df_bills['num_sub1'].apply(sub_num)
df_bills = df_bills.drop(columns=['num_sub1']) # this was a temporary column; i dont want to store it LOL




# attaching unemployment rate, real gdp variables and general fund rev nominal
new_vars = ['unemp_rate','real_gdp_chg', 'real_gdp_pct']
gfund_nom = {2016:7822419431, 2017:8103404448, 2018:8603652799, 2019:9385245545,
             2020:9965081246, 2021:13077188159, 2022:15062940729, 2023:15568599887,
             2024:16107674356, 2025:16208251717}

for year in np.arange(2016, 2025, 1):
    mask_bills = df_bills['session_year'] == year
    mask_fred = df_fred['observation_date'] == year

    # adding new_vars
    values = df_fred.loc[mask_fred, new_vars].values[0]
    df_bills.loc[mask_bills, new_vars] = values

# adding gnom_fund
mask = df_bills['session_year'].between(2016, 2025)
df_bills.loc[mask, 'gfund_nom'] = (df_bills.loc[mask, 'session_year'].map(gfund_nom)).astype(int)
df_bills.loc[mask, 'gfund_real'] = (df_bills.loc[mask, 'gfund_nom'] * df_bills.loc[mask, 'session_year'].map(inflation)
                                   ).astype(int)


# extracting and summing money amounts
def money_sub(money_big):
    if pd.isna(money_big):
        return 0
    if money_big == 'None':
        return 0
    cells = [cell.strip() for cell in money_big.split('<hr>')]
    money_values = []
    for cell in cells:
        match = re.search(r'\$([0-9,]+)', cell)
        if match:
            money_values.append(int(re.search(r'\$([0-9,]+)', cell).group(1).replace(',','')))
            return sum(money_values)
        else:
            return 0

df_bills['money_sum'] = df_bills['money_appropriated'].apply(money_sub)

# inflation adjusting money_sum
mask = df_bills['session_year'].between(2016,2026)
df_bills.loc[mask, 'money_sum_real'] = (
    df_bills.loc[mask, 'money_sum'] * df_bills.loc[mask, 'session_year'].map(inflation)
).astype(int)



# adding party information
house1_df = pd.read_csv('house1_df.csv')
rep_dict = dict(zip(house1_df['id'], house1_df['party']))
df_bills['party'] = df_bills['sponsor_id'].map(rep_dict)
df_bills['party1'] = df_bills['party'].map(lambda x: 1 if x == 'Rep' else 0)



# dropping the unecessary columns
df_bills = df_bills.drop(columns = ['gfund_nom', 'money_sum'])

print(df_bills.shape)
df_bills.head()

(8705, 19) (9533, 19)
(8705, 46)


,bill_id,short_title,sponsor,co_sponsors,floor_sponsor,last_action,last_action_owner,subjects,num_substitutes,substitute_sponsors,...,money1,cospon_num,num_sub_new,unemp_rate,real_gdp_chg,real_gdp_pct,gfund_real,money_sum_real,party,party1
0,HB0001,Public Education Base Budget Amendments,ELIASS,None,STEPHHA,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Public Education; Appropriations,0,NaN,...,1,0,1,3.3,7096.7,4.56581,1.049292e+10,670695.0,Rep,1
1,HB0002,New Fiscal Year Supplemental Appropriations Act,SANPED,None,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,...,1,0,0,3.3,7096.7,4.56581,1.049292e+10,724863681.0,Rep,1
2,HB0003,Appropriations Adjustments,SANPED,None,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,...,1,0,0,3.3,7096.7,4.56581,1.049292e+10,7266980.0,Rep,1
3,HB0005,"Natural Resources, Agriculture, and Environmen...",MCKELMK,None,HINKIDP,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Natural Resources; Appropriations,0,NaN,...,1,0,0,3.3,7096.7,4.56581,1.049292e+10,429675922.0,Rep,1
4,HB0006,Executive Offices and Criminal Justice Base Bu...,HUTCHEK,None,THATCDW,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Law Enforcement and Criminal Justic...,0,NaN,...,1,0,1,3.3,7096.7,4.56581,1.049292e+10,829247.0,Rep,1


In [64]:
df_bills.to_csv('df_bills.csv', index = False)
df_bills.head()

,bill_id,short_title,sponsor,co_sponsors,floor_sponsor,last_action,last_action_owner,subjects,num_substitutes,substitute_sponsors,...,money1,cospon_num,num_sub_new,unemp_rate,real_gdp_chg,real_gdp_pct,gfund_real,money_sum_real,party,party1
0,HB0001,Public Education Base Budget Amendments,ELIASS,None,STEPHHA,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Public Education; Appropriations,0,NaN,...,1,0,1,3.3,7096.7,4.56581,1.049292e+10,670695.0,Rep,1
1,HB0002,New Fiscal Year Supplemental Appropriations Act,SANPED,None,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,...,1,0,0,3.3,7096.7,4.56581,1.049292e+10,724863681.0,Rep,1
2,HB0003,Appropriations Adjustments,SANPED,None,HILLYLW,Governor Line Item Veto,Lieutenant Governor's office for filing,Budgeting; Appropriations,0,NaN,...,1,0,0,3.3,7096.7,4.56581,1.049292e+10,7266980.0,Rep,1
3,HB0005,"Natural Resources, Agriculture, and Environmen...",MCKELMK,None,HINKIDP,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Natural Resources; Appropriations,0,NaN,...,1,0,0,3.3,7096.7,4.56581,1.049292e+10,429675922.0,Rep,1
4,HB0006,Executive Offices and Criminal Justice Base Bu...,HUTCHEK,None,THATCDW,Governor Signed,Lieutenant Governor's office for filing,Budgeting; Law Enforcement and Criminal Justic...,0,NaN,...,1,0,1,3.3,7096.7,4.56581,1.049292e+10,829247.0,Rep,1


In [67]:
for col in df_bills.columns:
    print(col)

bill_id
short_title
sponsor
co_sponsors
floor_sponsor
last_action
last_action_owner
subjects
num_substitutes
substitute_sponsors
active_version
subjects_changed
committees
money_appropriated
originating_chamber
final_chamber
sponsor_id
session_year
session_id
location_stage
stage_1
stage_2
stage_3
stage_4
passed
subject_group
educ
cj_civ
econ
health
other
gov_ops
env_nr
committee1
cospon1
floorspon1
money1
cospon_num
num_sub_new
unemp_rate
real_gdp_chg
real_gdp_pct
gfund_real
money_sum_real
party
party1


# Exploration
TO DO:
* Look at distributions of key variables: boxplots, scatterplots, PDFs
* Get correlation/colinearity of subjects: scatter matrix
* Decide on GDP and unemp variables

In [14]:
# importing packages
import matplotlib.pyplot as plt
%matplotlib inline

In [75]:
# masks for desired variables
outcomes = ['stage_1', 'stage_2', 'stage_3', 'stage_4', 'passed']
subjects = list(set(subject_to_group.values()))
one_hots = ['committee1', 'cospon1', 'floorspon1', 'money1']
misc = ['cospon_num', 'num_sub_new', 'num_sub', 'money_sum_real']
econ = ['unemp_rate', 'real_gdp', 'read_gdp_pct', 'gfund_real']

Looks like a lot of bills die in stage 0 (makes sense), but if they get past that first stage, they (supposedly) will be more likely to pass than die in the intermediate stages.

In [44]:
# outcomes shapes
print(df_bills[outcomes].describe())

           stage_1      stage_2      stage_3      stage_4       passed
count  8705.000000  8705.000000  8705.000000  8705.000000  8705.000000
mean      0.379322     0.001034     0.005284     0.614360     0.614360
std       0.485246     0.032139     0.072505     0.486774     0.486774
min       0.000000     0.000000     0.000000     0.000000     0.000000
25%       0.000000     0.000000     0.000000     0.000000     0.000000
50%       0.000000     0.000000     0.000000     1.000000     1.000000
75%       1.000000     0.000000     0.000000     1.000000     1.000000
max       1.000000     1.000000     1.000000     1.000000     1.000000


In [93]:
# subjects
for subj in subjects:
    total_bills = df_bills[subj].value_counts()[1]
    total_pass = df_bills.loc[df_bills['passed']==1, subj].value_counts()[1]
    print(f'\n===={subj}====')
    print(f'Total Bills: {total_bills}')
    print(f'Total Bills Passed: {total_pass}')
    print(f'Percent Bills Passed: {round((total_pass/total_bills)*100, 2)}%')


====educ====
Total Bills: 985
Total Bills Passed: 567
Percent Bills Passed: 57.56%

====cj_civ====
Total Bills: 1141
Total Bills Passed: 690
Percent Bills Passed: 60.47%

====econ====
Total Bills: 1303
Total Bills Passed: 805
Percent Bills Passed: 61.78%

====health====
Total Bills: 1414
Total Bills Passed: 877
Percent Bills Passed: 62.02%

====other====
Total Bills: 1004
Total Bills Passed: 657
Percent Bills Passed: 65.44%

====gov_ops====
Total Bills: 2100
Total Bills Passed: 1255
Percent Bills Passed: 59.76%

====env_nr====
Total Bills: 758
Total Bills Passed: 497
Percent Bills Passed: 65.57%
